In [1]:
import os
import json
import xml.etree.ElementTree as ET

import epo_ops
import spacy
from dotenv import load_dotenv
from PatentProvider import PatentProvider
from kg.formatting.formatting_manager import FormattingManager



c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
import json

INPUT_EXPORT = "training/ner/18_12_ner_class_ls.json"
OUTPUT_DATASET = "training/ner/ner_classification_dataset_training.jsonl"

LABELS = [
    "INVENTION","COMPONENT","SUBSYSTEM","MATERIAL","CHEMICAL","BIOMOLECULE","COMPOSITION",
    "PROCESS_STEP","METHOD","PARAMETER","MEASUREMENT","CONDITION","FUNCTION","SIGNAL",
    "CONTROL","SOFTWARE","HARDWARE","FIGURE_REF","CLAIM_ELEMENT","PRIOR_ART","UNCLASSIFIED_ENTITY",
]
label_set = set(LABELS)

with open(INPUT_EXPORT, "r", encoding="utf-8") as f:
    tasks = json.load(f)

written = 0
skipped_no_text = 0
skipped_no_anno = 0
skipped_no_ents = 0

with open(OUTPUT_DATASET, "w", encoding="utf-8") as out:
    for idx, task in enumerate(tasks, start=1):
        text = (task.get("data", {}).get("text") or "").strip()
        if not text:
            skipped_no_text += 1
            continue

        annotations = task.get("annotations") or task.get("completions") or []
        if not annotations:
            skipped_no_anno += 1
            continue

        # take first annotation by default (same as your original)
        results = annotations[0].get("result", [])
        entities = []

        for r in results:
            value = r.get("value", {}) or {}
            start = value.get("start")
            end = value.get("end")
            labels = value.get("labels")

            if start is None or end is None or not labels:
                continue

            label = labels[0]
            if label not in label_set:
                continue

            s = int(start)
            e = int(end)

            # basic sanity: must be inside text and non-empty
            if s < 0 or e <= s or e > len(text):
                continue

            # ✅ Arrow-friendly struct entity
            entities.append({"start": s, "end": e, "label": label})

        if not entities:
            skipped_no_ents += 1
            continue

        entities.sort(key=lambda x: (x["start"], x["end"], x["label"]))

        # ensure id is numeric-ish; fall back to running index
        raw_id = task.get("id", idx)
        try:
            _id = int(raw_id)
        except Exception:
            _id = idx

        out.write(json.dumps(
            {"id": _id, "text": text, "entities": entities},
            ensure_ascii=False
        ) + "\n")
        written += 1

print(f"JSONL file written to: {OUTPUT_DATASET}")
print(f"Written: {written}")
print(f"Skipped (no text): {skipped_no_text}")
print(f"Skipped (no annotation): {skipped_no_anno}")
print(f"Skipped (no valid entities): {skipped_no_ents}")


JSONL file written to: training/ner/ner_classification_dataset_training.jsonl
Written: 11962
Skipped (no text): 0
Skipped (no annotation): 0
Skipped (no valid entities): 1040


In [1]:
pip install -U transformers datasets accelerate evaluate seqeval


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl (2.7 MB)
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16283 sha256=a5cf436304acb0475692663210220cfc4f4a97b021994747a70f4c3ab9f114db
  Stored in directory: c:\users\caleb\appdata\local\pip\cache\wheels\14\cf\a7\8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully bu

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.79.1 requires jinja2<4.0.0,>=3.1.2, but you have jinja2 3.0.3 which is incompatible.
spacy-transformers 1.3.9 requires transformers<4.50.0,>=3.4.0, but you have transformers 4.57.3 which is incompatible.


In [18]:
import json
import numpy as np
from pathlib import Path

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate


DATA_PATH = "training/ner/ner_classification_dataset_training.jsonl"
OUT_DIR = "training/ner/done/hf/ner_model"
MODEL_NAME = "anferico/bert-for-patents"


# -------------------------
# 1) Read JSONL
# -------------------------
def read_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}\n{line[:200]}") from e

            if "text" not in obj or "entities" not in obj:
                raise ValueError(f"Missing fields on line {line_no}: {obj.keys()}")

            # entities now must be list of dicts: {"start": int, "end": int, "label": str}
            if not isinstance(obj["entities"], list):
                raise ValueError(f"'entities' is not a list on line {line_no}")

            rows.append(obj)
    return rows


rows = read_jsonl(DATA_PATH)
ds = Dataset.from_list(rows).train_test_split(test_size=0.2, seed=42)
train_ds = ds["train"]
eval_ds = ds["test"]


# -------------------------
# 2) Build label set from entities (dict format)
# -------------------------
def get_entity_types(dataset):
    types = set()
    for ents in dataset["entities"]:
        if isinstance(ents, list):
            for ent in ents:
                if isinstance(ent, dict) and isinstance(ent.get("label"), str):
                    types.add(ent["label"])
    return sorted(types)

entity_types = get_entity_types(train_ds)
label_list = ["O"] + [tag for t in entity_types for tag in (f"B-{t}", f"I-{t}")]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print("Entity types:", entity_types)
print("Num labels:", len(label_list))


# -------------------------
# 3) Tokenize + align char spans to token labels (dict format)
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def encode_batch(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        return_offsets_mapping=True,
    )

    all_labels = []
    for text, ents, offsets in zip(examples["text"], examples["entities"], enc["offset_mapping"]):
        token_tags = ["O"] * len(offsets)

        # ents: [{"start":..,"end":..,"label":..}, ...]
        ents = sorted(ents, key=lambda x: x.get("start", 0)) if isinstance(ents, list) else []

        for ent in ents:
            if not isinstance(ent, dict):
                continue

            start = ent.get("start")
            end = ent.get("end")
            ent_type = ent.get("label")

            if start is None or end is None or ent_type is None:
                continue

            start = int(start)
            end = int(end)
            ent_type = str(ent_type)

            first = True
            for i, (ts, te) in enumerate(offsets):
                # skip special tokens (often offset (0,0))
                if ts == te == 0:
                    continue
                # token overlaps entity?
                if ts < end and te > start:
                    token_tags[i] = f"B-{ent_type}" if first else f"I-{ent_type}"
                    first = False

        # tag -> id, mask specials
        label_ids = []
        for (ts, te), tag in zip(offsets, token_tags):
            label_ids.append(-100 if ts == te == 0 else label2id[tag])

        all_labels.append(label_ids)

    enc.pop("offset_mapping")
    enc["labels"] = all_labels
    return enc

train_tok = train_ds.map(encode_batch, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(encode_batch,  batched=True, remove_columns=eval_ds.column_names)


# -------------------------
# 4) Train
# -------------------------
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pr, lb in zip(preds, labels):
        p_seq, l_seq = [], []
        for p_id, l_id in zip(pr, lb):
            if l_id == -100:
                continue
            p_seq.append(id2label[int(p_id)])
            l_seq.append(id2label[int(l_id)])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    r = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": r["overall_precision"],
        "recall": r["overall_recall"],
        "f1": r["overall_f1"],
        "accuracy": r["overall_accuracy"],
    }

args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=False,  # set True if you have CUDA
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

print(f"✅ Saved to {OUT_DIR}")


Entity types: ['BIOMOLECULE', 'CHEMICAL', 'CLAIM_ELEMENT', 'COMPONENT', 'COMPOSITION', 'CONDITION', 'CONTROL', 'FIGURE_REF', 'FUNCTION', 'HARDWARE', 'INVENTION', 'MATERIAL', 'MEASUREMENT', 'METHOD', 'PARAMETER', 'PRIOR_ART', 'PROCESS_STEP', 'SIGNAL', 'SOFTWARE', 'SUBSYSTEM', 'UNCLASSIFIED_ENTITY']
Num labels: 43


Map: 100%|██████████| 2393/2393 [00:00<00:00, 16449.52 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at anferico/bert-for-patents and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Caleb\AppData\Local\Temp\ipykernel_382668\1921945488.py:187: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.413200,0.454558,0.615614,0.657240,0.635746,0.863731
2,0.311200,0.389818,0.680592,0.691097,0.685805,0.884103
3,0.209100,0.391085,0.688498,0.712906,0.700489,0.887794


✅ Saved to training/ner/done/hf/ner_model
